# ML Models — AI Hype & Stock Returns

This notebook applies machine learning methods to the merged dataset from Milestone 1. The goal is to go beyond correlation and hypothesis testing — I want to see whether AI hype signals can actually **predict** stock behavior, and what patterns emerge when we let the data cluster itself.

Methods covered:
- Linear Regression (OLS, Ridge, Lasso) — for all three tickers
- Logistic Regression
- K-Nearest Neighbors
- Decision Tree
- Ensemble: Bagging, Random Forest, Gradient Boosted Trees, Voting Committee
- Unsupervised: K-Means Clustering, Hierarchical Clustering

All classification models are evaluated with the same stratified 5-fold CV setup so the numbers are directly comparable.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

from sklearn.model_selection import (
    train_test_split, cross_val_score,
    StratifiedKFold, cross_validate
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    BaggingClassifier
)
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

PROJECT_ROOT   = Path("/Users/demirhanmac/DSA210-TermProject-Hasan-Demirhan-Durmaz")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES        = PROJECT_ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

RANDOM_STATE = 42
TICKERS      = ["NVDA", "MSFT", "AAPL"]
CV           = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

plt.rcParams["figure.dpi"]        = 120
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

print("imports ok")

## 1. Load & Feature Engineering

I load the merged dataset from Milestone 1 and engineer a few extra features that should help the ML models: lagged hype (yesterday's hype predicting today's return), rolling hype momentum, and 5-day forward returns for each ticker as regression targets.

In [ ]:
df = pd.read_csv(DATA_PROCESSED / "merged_data.csv", index_col="date", parse_dates=True)
print(f"loaded: {df.shape}  |  {df.index.min().date()} to {df.index.max().date()}")
print(f"columns: {list(df.columns)}")

In [ ]:
# lagged hype (1, 5, 20 trading days)
for lag in [1, 5, 20]:
    df[f"hype_lag{lag}"] = df["hype_index"].shift(lag)

# 5-day rolling hype momentum
df["hype_momentum_5d"] = df["hype_index"].diff(5)

# 5-day forward return for each ticker (regression target)
for ticker in TICKERS:
    df[f"{ticker}_fwd5"] = df[f"{ticker}_return"].shift(-5).rolling(5).mean()

df = df.dropna()
print(f"after feature engineering: {df.shape}")
print(f"\nis_hype_period distribution:\n{df['is_hype_period'].value_counts()}")

In [ ]:
HYPE_FEATURES = [
    "hype_index", "google_hype_index", "reddit_hype_index",
    "hype_lag1", "hype_lag5", "hype_lag20", "hype_momentum_5d"
]

FULL_FEATURES = HYPE_FEATURES + [
    "NVDA_return", "MSFT_return", "AAPL_return",
    "NVDA_volatility", "MSFT_volatility", "AAPL_volatility"
]

# train/test split — chronological, no shuffle
split_idx = int(len(df) * 0.8)
train_df  = df.iloc[:split_idx]
test_df   = df.iloc[split_idx:]

print(f"train: {train_df.index.min().date()} → {train_df.index.max().date()}  ({len(train_df)} days)")
print(f"test:  {test_df.index.min().date()} → {test_df.index.max().date()}  ({len(test_df)} days)")

## 2. Linear Regression — Hype as a Predictor of Returns

I start with the simplest possible question: can hype features directly predict 5-day forward returns? I compare OLS, Ridge, and Lasso for all three tickers so we can see whether the relationship differs across companies. NVDA is the most directly tied to AI infrastructure so I expect it to show the strongest (though still weak) signal.

In [ ]:
scaler_reg = StandardScaler()
X_tr_reg = scaler_reg.fit_transform(train_df[HYPE_FEATURES])
X_te_reg = scaler_reg.transform(test_df[HYPE_FEATURES])

regressors = {
    "OLS":   LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.0001, max_iter=5000),
}

reg_results = []
for ticker in TICKERS:
    y_tr = train_df[f"{ticker}_fwd5"]
    y_te = test_df[f"{ticker}_fwd5"]
    for name, model in regressors.items():
        model.fit(X_tr_reg, y_tr)
        pred = model.predict(X_te_reg)
        rmse = np.sqrt(mean_squared_error(y_te, pred))
        r2   = r2_score(y_te, pred)
        reg_results.append({"ticker": ticker, "model": name,
                             "RMSE": round(rmse, 6), "R²": round(r2, 4)})

reg_df = pd.DataFrame(reg_results)
print("Regression results (target: 5-day forward return)")
print("-" * 55)
display(reg_df.pivot(index="model", columns="ticker", values="R²"))

In [ ]:
# OLS coefficients for all three tickers
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, ticker in zip(axes, TICKERS):
    ols = LinearRegression()
    ols.fit(X_tr_reg, train_df[f"{ticker}_fwd5"])
    coef = pd.Series(ols.coef_, index=HYPE_FEATURES).sort_values()
    colors = ["#d62728" if c < 0 else "#2ca02c" for c in coef.values]
    coef.plot(kind="barh", ax=ax, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{ticker} — OLS coefficients")
    ax.set_xlabel("coefficient (standardized)")

fig.suptitle("Which Hype Features Drive Each Ticker's Forward Return?", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES / "10_ols_coefficients.png", bbox_inches="tight")
plt.show()

print("\nInterpretation:")
print("R² values are very low across all tickers — hype features alone explain")
print("little variance in forward returns. This is consistent with market efficiency:")
print("if hype were a clean predictor, arbitrageurs would eliminate the signal.")
print("NVDA tends to show the strongest (though still weak) relationship, which makes")
print("sense given its direct exposure to AI hardware demand.")

In [ ]:
# Lasso feature selection — which features survive regularization?
print("Lasso feature selection per ticker (alpha=0.0001)")
print("-" * 50)
for ticker in TICKERS:
    lasso = Lasso(alpha=0.0001, max_iter=5000)
    lasso.fit(X_tr_reg, train_df[f"{ticker}_fwd5"])
    surviving = [f for f, c in zip(HYPE_FEATURES, lasso.coef_) if abs(c) > 1e-6]
    zeroed    = [f for f, c in zip(HYPE_FEATURES, lasso.coef_) if abs(c) <= 1e-6]
    print(f"\n{ticker}:")
    print(f"  kept:   {surviving}")
    print(f"  zeroed: {zeroed}")

## 3. Classification Setup

For the rest of the notebook the task switches to classification: predict whether a day is a **high-hype period** (`is_hype_period = 1`, top 25% of hype index). All models use the same stratified 5-fold CV and the same train/test split, so results are directly comparable.

In [ ]:
X_tr_c = train_df[FULL_FEATURES]
y_tr_c = train_df["is_hype_period"]
X_te_c = test_df[FULL_FEATURES]
y_te_c = test_df["is_hype_period"]

print(f"train class balance: {y_tr_c.value_counts().to_dict()}")
print(f"test  class balance: {y_te_c.value_counts().to_dict()}")

# helper: train, CV, and evaluate a classifier
def evaluate_classifier(name, pipeline, X_tr, y_tr, X_te, y_te, cv=CV):
    cv_res = cross_validate(pipeline, X_tr, y_tr, cv=cv,
                            scoring=["accuracy", "roc_auc"], return_train_score=False)
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    y_prob = pipeline.predict_proba(X_te)[:, 1]
    return {
        "Model":        name,
        "CV Acc (mean)": round(cv_res["test_accuracy"].mean(), 4),
        "CV Acc (std)":  round(cv_res["test_accuracy"].std(),  4),
        "CV AUC (mean)": round(cv_res["test_roc_auc"].mean(),  4),
        "CV AUC (std)":  round(cv_res["test_roc_auc"].std(),   4),
        "Test Acc":      round(accuracy_score(y_te, y_pred),   4),
        "Test AUC":      round(roc_auc_score(y_te, y_prob),    4),
        "_pipeline":     pipeline,
        "_y_pred":       y_pred,
        "_y_prob":       y_prob,
    }

all_results = []  # collect here
print("setup done")

## 4. Logistic Regression

In [ ]:
logreg_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, C=0.1, random_state=RANDOM_STATE)
)
lr_res = evaluate_classifier("Logistic Regression", logreg_pipe, X_tr_c, y_tr_c, X_te_c, y_te_c)
all_results.append(lr_res)

print(f"Logistic Regression")
print(f"  CV  Accuracy: {lr_res['CV Acc (mean)']:.4f} ± {lr_res['CV Acc (std)']:.4f}")
print(f"  CV  AUC:      {lr_res['CV AUC (mean)']:.4f} ± {lr_res['CV AUC (std)']:.4f}")
print(f"  Test Accuracy: {lr_res['Test Acc']:.4f}")
print(f"  Test AUC:      {lr_res['Test AUC']:.4f}")
print()
print(classification_report(y_te_c, lr_res["_y_pred"], target_names=["normal", "hype"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_te_c, lr_res["_y_pred"])
ConfusionMatrixDisplay(cm, display_labels=["normal", "hype"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Logistic Regression — Confusion Matrix")

fpr, tpr, _ = roc_curve(y_te_c, lr_res["_y_prob"])
axes[1].plot(fpr, tpr, lw=2, label=f"AUC = {lr_res['Test AUC']:.3f}")
axes[1].plot([0,1],[0,1], "k--", lw=1)
axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve — Logistic Regression"); axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / "11_logreg_results.png", bbox_inches="tight")
plt.show()

print("\nInterpretation:")
print("Logistic regression gives a solid AUC with a linear decision boundary.")
print("The confusion matrix shows it correctly identifies most hype periods,")
print("but with some false negatives — days the model calls 'normal' that were")
print("actually high-hype. This makes sense: linear models miss the interaction")
print("effects between volatility and hype that tree-based models can capture.")

## 5. K-Nearest Neighbors

KNN makes no assumptions about the data distribution — it just finds the k most similar past trading days and votes. The key tuning parameter is k; too small means overfitting to noise, too large smooths out the signal.

In [ ]:
scaler_knn = StandardScaler()
X_tr_knn   = scaler_knn.fit_transform(X_tr_c)
X_te_knn   = scaler_knn.transform(X_te_c)

k_range    = range(3, 32, 2)
cv_acc     = []
cv_auc     = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    res = cross_validate(knn, X_tr_knn, y_tr_c, cv=CV,
                         scoring=["accuracy", "roc_auc"])
    cv_acc.append(res["test_accuracy"].mean())
    cv_auc.append(res["test_roc_auc"].mean())

best_k = list(k_range)[np.argmax(cv_auc)]
print(f"best k = {best_k}  (CV AUC = {max(cv_auc):.4f})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), cv_acc, marker="o", color="#1f77b4")
axes[0].axvline(best_k, color="red", linestyle="--", label=f"best k={best_k}")
axes[0].set_xlabel("k"); axes[0].set_ylabel("CV Accuracy")
axes[0].set_title("KNN — CV Accuracy vs k"); axes[0].legend()

axes[1].plot(list(k_range), cv_auc, marker="s", color="#ff7f0e")
axes[1].axvline(best_k, color="red", linestyle="--", label=f"best k={best_k}")
axes[1].set_xlabel("k"); axes[1].set_ylabel("CV AUC")
axes[1].set_title("KNN — CV AUC vs k"); axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / "12_knn_sweep.png", bbox_inches="tight")
plt.show()

In [ ]:
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_tr_knn, y_tr_c)
y_pred_knn = knn_best.predict(X_te_knn)
y_prob_knn = knn_best.predict_proba(X_te_knn)[:, 1]

knn_entry = {
    "Model":         f"KNN (k={best_k})",
    "CV Acc (mean)": round(max(cv_acc), 4),
    "CV Acc (std)":  None,
    "CV AUC (mean)": round(max(cv_auc), 4),
    "CV AUC (std)":  None,
    "Test Acc":      round(accuracy_score(y_te_c, y_pred_knn), 4),
    "Test AUC":      round(roc_auc_score(y_te_c, y_prob_knn), 4),
    "_y_pred":       y_pred_knn,
    "_y_prob":       y_prob_knn,
}
all_results.append(knn_entry)

print(f"KNN (k={best_k}) — Test Accuracy: {knn_entry['Test Acc']}  |  Test AUC: {knn_entry['Test AUC']}")
print()
print(classification_report(y_te_c, y_pred_knn, target_names=["normal", "hype"]))

print("\nInterpretation:")
print("KNN performance depends heavily on k. Small k overfit to noisy daily returns;")
print("larger k starts generalizing better. The chosen k reflects a bias-variance")
print("tradeoff visible in the sweep above.")

## 6. Decision Tree

Decision trees split the feature space into rectangular regions using simple if-then rules. They are fully interpretable — we can read exactly what the model learned. The main risk is overfitting, which I control by cross-validating tree depth.

In [ ]:
depth_scores = {}
for d in range(2, 13):
    dt   = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    res  = cross_validate(dt, X_tr_c, y_tr_c, cv=CV, scoring=["accuracy", "roc_auc"])
    depth_scores[d] = {
        "acc": res["test_accuracy"].mean(),
        "auc": res["test_roc_auc"].mean()
    }

best_depth = max(depth_scores, key=lambda d: depth_scores[d]["auc"])
print(f"best depth = {best_depth}  (CV AUC = {depth_scores[best_depth]['auc']:.4f})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
depths = list(depth_scores.keys())
axes[0].plot(depths, [depth_scores[d]["acc"] for d in depths], marker="o", color="#1f77b4")
axes[0].axvline(best_depth, color="red", linestyle="--", label=f"best={best_depth}")
axes[0].set_xlabel("max_depth"); axes[0].set_ylabel("CV Accuracy")
axes[0].set_title("Decision Tree — CV Accuracy vs Depth"); axes[0].legend()

axes[1].plot(depths, [depth_scores[d]["auc"] for d in depths], marker="s", color="#ff7f0e")
axes[1].axvline(best_depth, color="red", linestyle="--", label=f"best={best_depth}")
axes[1].set_xlabel("max_depth"); axes[1].set_ylabel("CV AUC")
axes[1].set_title("Decision Tree — CV AUC vs Depth"); axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / "13_dt_depth_sweep.png", bbox_inches="tight")
plt.show()

In [ ]:
dt_pipe = DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE)
dt_res  = evaluate_classifier(f"Decision Tree (d={best_depth})", dt_pipe, X_tr_c, y_tr_c, X_te_c, y_te_c)
all_results.append(dt_res)

print(f"Decision Tree (depth={best_depth})")
print(f"  CV  Accuracy: {dt_res['CV Acc (mean)']:.4f} ± {dt_res['CV Acc (std)']:.4f}")
print(f"  CV  AUC:      {dt_res['CV AUC (mean)']:.4f} ± {dt_res['CV AUC (std)']:.4f}")
print(f"  Test Accuracy: {dt_res['Test Acc']:.4f}")
print(f"  Test AUC:      {dt_res['Test AUC']:.4f}")
print()
print(classification_report(y_te_c, dt_res["_y_pred"], target_names=["normal", "hype"]))

# visualize the tree (top 3 levels)
fig, ax = plt.subplots(figsize=(15, 6))
plot_tree(
    dt_pipe, max_depth=3, feature_names=FULL_FEATURES,
    class_names=["normal", "hype"], filled=True, ax=ax, fontsize=7.5
)
ax.set_title(f"Decision Tree (top 3 levels, full depth={best_depth})", fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / "14_decision_tree.png", bbox_inches="tight")
plt.show()

print("\nInterpretation:")
print("The first splits reveal which features the tree considers most informative.")
print("Splits near the root on volatility or lagged hype confirm that hype periods")
print("are associated with elevated market activity — consistent with Milestone 1.")

## 7. Ensemble Methods

### 7.1 Bagging

Bagging (Bootstrap Aggregating) trains many trees on random subsamples of the training data and averages their votes. This reduces variance without increasing bias — each individual tree overfits slightly, but the average is much more stable.

In [ ]:
bag_pipe = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=best_depth),
    n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
)
bag_res = evaluate_classifier("Bagging", bag_pipe, X_tr_c, y_tr_c, X_te_c, y_te_c)
all_results.append(bag_res)

print(f"Bagging (100 trees)")
print(f"  CV  Accuracy: {bag_res['CV Acc (mean)']:.4f} ± {bag_res['CV Acc (std)']:.4f}")
print(f"  CV  AUC:      {bag_res['CV AUC (mean)']:.4f} ± {bag_res['CV AUC (std)']:.4f}")
print(f"  Test Accuracy: {bag_res['Test Acc']:.4f}  |  Test AUC: {bag_res['Test AUC']:.4f}")

print("\nInterpretation:")
print("Bagging already improves on a single decision tree. Averaging 100 bootstrapped")
print("trees smooths out the variance that made the single tree unstable.")

### 7.2 Random Forest

Random Forest extends bagging by also randomizing which features each tree can split on at each node. This decorrelates the trees — if one feature dominates, bagging would still have correlated trees all splitting on it. Random Forest avoids this and typically reduces variance further.

In [ ]:
rf_pipe = RandomForestClassifier(
    n_estimators=200, max_depth=best_depth + 2,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_res = evaluate_classifier("Random Forest", rf_pipe, X_tr_c, y_tr_c, X_te_c, y_te_c)
all_results.append(rf_res)

print(f"Random Forest (200 trees)")
print(f"  CV  Accuracy: {rf_res['CV Acc (mean)']:.4f} ± {rf_res['CV Acc (std)']:.4f}")
print(f"  CV  AUC:      {rf_res['CV AUC (mean)']:.4f} ± {rf_res['CV AUC (std)']:.4f}")
print(f"  Test Accuracy: {rf_res['Test Acc']:.4f}  |  Test AUC: {rf_res['Test AUC']:.4f}")
print()
print(classification_report(y_te_c, rf_res["_y_pred"], target_names=["normal", "hype"]))

In [ ]:
importances = pd.Series(rf_pipe.feature_importances_, index=FULL_FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
importances.plot(kind="bar", ax=ax, color="#1f77b4")
ax.set_title("Random Forest — Feature Importances (Gini)", fontsize=11)
ax.set_ylabel("importance")
ax.set_xticklabels(importances.index, rotation=40, ha="right")
plt.tight_layout()
plt.savefig(FIGURES / "15_rf_importances.png", bbox_inches="tight")
plt.show()

print("\nTop 5 features:")
print(importances.head(5).to_string())
print("\nInterpretation:")
print("The feature importance ranking tells us what the forest actually relies on.")
print("If volatility features dominate alongside hype_index, it confirms that hype")
print("periods are fundamentally high-volatility regimes, not just high-return ones.")
print("Lagged hype features appearing in the top 5 would suggest some persistence")
print("in the hype signal — today's hype predicts tomorrow's regime.")

### 7.3 Gradient Boosted Decision Trees

Gradient Boosting builds trees sequentially: each new tree is trained on the residual errors of the previous ensemble. This is fundamentally different from bagging — instead of parallel independent trees, it's a chain where each model corrects the last. It often achieves the best performance but requires careful tuning of learning rate and depth to avoid overfitting.

In [ ]:
gbt_pipe = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05,
    max_depth=4, subsample=0.8, random_state=RANDOM_STATE
)
gbt_res = evaluate_classifier("Gradient Boosting", gbt_pipe, X_tr_c, y_tr_c, X_te_c, y_te_c)
all_results.append(gbt_res)

print(f"Gradient Boosted Trees")
print(f"  CV  Accuracy: {gbt_res['CV Acc (mean)']:.4f} ± {gbt_res['CV Acc (std)']:.4f}")
print(f"  CV  AUC:      {gbt_res['CV AUC (mean)']:.4f} ± {gbt_res['CV AUC (std)']:.4f}")
print(f"  Test Accuracy: {gbt_res['Test Acc']:.4f}  |  Test AUC: {gbt_res['Test AUC']:.4f}")
print()
print(classification_report(y_te_c, gbt_res["_y_pred"], target_names=["normal", "hype"]))

In [ ]:
# learning curve — how does AUC evolve as more trees are added?
train_aucs, test_aucs = [], []
for y_prob_tr in gbt_pipe.staged_predict_proba(X_tr_c):
    train_aucs.append(roc_auc_score(y_tr_c, y_prob_tr[:, 1]))
for y_prob_te in gbt_pipe.staged_predict_proba(X_te_c):
    test_aucs.append(roc_auc_score(y_te_c, y_prob_te[:, 1]))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_aucs, label="train AUC", lw=2, color="#1f77b4")
ax.plot(test_aucs,  label="test AUC",  lw=2, color="#ff7f0e")
best_iter = int(np.argmax(test_aucs))
ax.axvline(best_iter, color="red", linestyle="--", label=f"best iter={best_iter}")
ax.set_xlabel("number of trees")
ax.set_ylabel("ROC-AUC")
ax.set_title("Gradient Boosting — Learning Curve")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "16_gbt_learning_curve.png", bbox_inches="tight")
plt.show()

print(f"\nBest test AUC at iteration {best_iter}: {test_aucs[best_iter]:.4f}")
print("\nInterpretation:")
print("The gap between train and test AUC shows overfitting dynamics. A wide gap")
print("means the model is memorizing the training data. The learning rate of 0.05")
print("and subsample=0.8 are deliberately conservative to keep the gap small.")

### 7.4 Voting Ensemble (Simple Committee)

A committee that combines logistic regression, random forest, and gradient boosting via soft voting. Each model outputs a probability and the final prediction is the average. The idea is that different models make different errors, and averaging reduces total error.

In [ ]:
voting_pipe = VotingClassifier(
    estimators=[
        ("lr",  make_pipeline(StandardScaler(),
                              LogisticRegression(max_iter=1000, C=0.1, random_state=RANDOM_STATE))),
        ("rf",  RandomForestClassifier(n_estimators=200, max_depth=best_depth+2,
                                       random_state=RANDOM_STATE, n_jobs=-1)),
        ("gbt", GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                           max_depth=4, subsample=0.8, random_state=RANDOM_STATE)),
    ],
    voting="soft"
)
vote_res = evaluate_classifier("Voting Ensemble", voting_pipe, X_tr_c, y_tr_c, X_te_c, y_te_c)
all_results.append(vote_res)

print(f"Voting Ensemble (LR + RF + GBT)")
print(f"  CV  Accuracy: {vote_res['CV Acc (mean)']:.4f} ± {vote_res['CV Acc (std)']:.4f}")
print(f"  CV  AUC:      {vote_res['CV AUC (mean)']:.4f} ± {vote_res['CV AUC (std)']:.4f}")
print(f"  Test Accuracy: {vote_res['Test Acc']:.4f}  |  Test AUC: {vote_res['Test AUC']:.4f}")

print("\nInterpretation:")
print("If the ensemble beats all individual models, it confirms that LR, RF, and GBT")
print("are making partially independent errors. If it doesn't, it means the models")
print("are correlated enough that averaging doesn't help.")

## 8. Unsupervised Learning — Clustering

### 8.1 K-Means

Setting aside the labels entirely, I want to see what natural groupings emerge from the data. K-Means partitions the feature space into k clusters by minimizing within-cluster sum of squared distances. I use the elbow method and silhouette score to pick the right k.

In [ ]:
CLUSTER_FEATURES = [
    "hype_index", "google_hype_index", "reddit_hype_index",
    "NVDA_return", "NVDA_volatility"
]

scaler_cl = StandardScaler()
X_cl_sc   = scaler_cl.fit_transform(df[CLUSTER_FEATURES])

inertias   = []
sil_scores = []
K_RANGE    = range(2, 9)

for k in K_RANGE:
    km     = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_cl_sc)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cl_sc, labels))

best_k_cl = list(K_RANGE)[np.argmax(sil_scores)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(K_RANGE), inertias, marker="o", color="#1f77b4")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia (within-cluster SSE)")
axes[0].set_title("K-Means — Elbow Method")

axes[1].plot(list(K_RANGE), sil_scores, marker="s", color="#ff7f0e")
axes[1].axvline(best_k_cl, color="red", linestyle="--", label=f"best k={best_k_cl}")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("K-Means — Silhouette Score vs k"); axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / "17_kmeans_selection.png", bbox_inches="tight")
plt.show()
print(f"best k by silhouette: {best_k_cl}")

In [ ]:
km_final    = KMeans(n_clusters=best_k_cl, random_state=RANDOM_STATE, n_init=10)
df["kmeans_cluster"] = km_final.fit_predict(X_cl_sc)

print(f"K-Means (k={best_k_cl}) — Cluster Profiles")
print("-" * 60)
profile = df.groupby("kmeans_cluster")[CLUSTER_FEATURES + ["is_hype_period"]].mean().round(4)
display(profile)

PALETTE = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for cid in sorted(df["kmeans_cluster"].unique()):
    sub = df[df["kmeans_cluster"] == cid]
    axes[0].scatter(sub["hype_index"], sub["NVDA_return"],
                    s=8, alpha=0.45, color=PALETTE[cid % 4], label=f"Cluster {cid}")
axes[0].set_xlabel("Hype Index"); axes[0].set_ylabel("NVDA Daily Return")
axes[0].set_title(f"K-Means Clusters (k={best_k_cl}) — Hype vs Return")
axes[0].legend(markerscale=3)

hype_frac = df.groupby("kmeans_cluster")["is_hype_period"].mean()
hype_frac.plot(kind="bar", ax=axes[1],
               color=[PALETTE[i % 4] for i in hype_frac.index])
axes[1].set_xlabel("Cluster"); axes[1].set_ylabel("Fraction Hype-Period Days")
axes[1].set_title("Hype Composition per Cluster")
axes[1].set_xticklabels(hype_frac.index, rotation=0)

plt.tight_layout()
plt.savefig(FIGURES / "18_kmeans_clusters.png", bbox_inches="tight")
plt.show()

print("\nInterpretation:")
print("If the clusters separate cleanly by hype fraction, it means the data has")
print("a natural regime structure that the supervised labels also capture.")
print("Clusters with high hype fraction + high volatility are the 'AI bubble' regime.")

In [ ]:
# cluster assignment over time
fig, ax = plt.subplots(figsize=(13, 3))
for cid in sorted(df["kmeans_cluster"].unique()):
    mask = df["kmeans_cluster"] == cid
    ax.scatter(df.index[mask], df.loc[mask, "kmeans_cluster"],
               s=5, alpha=0.4, color=PALETTE[cid % 4], label=f"Cluster {cid}")
ax.set_yticks(sorted(df["kmeans_cluster"].unique()))
ax.set_ylabel("Cluster"); ax.set_title("K-Means Cluster Assignment Over Time")
ax.legend(loc="upper left", markerscale=3, ncol=best_k_cl)
plt.tight_layout()
plt.savefig(FIGURES / "19_cluster_timeline.png", bbox_inches="tight")
plt.show()

### 8.2 Hierarchical Clustering

Hierarchical clustering builds a tree of merges (dendrogram) rather than assigning all points to a fixed k upfront. This is useful because we can inspect the merge history to understand at what level the data naturally splits — instead of pre-committing to a number of clusters, we let the structure reveal itself.

In [ ]:
# subsample for dendrogram legibility (hierarchical on full 1000+ rows is unreadable)
sample_idx = np.linspace(0, len(X_cl_sc) - 1, 200, dtype=int)
X_hc_sample = X_cl_sc[sample_idx]

linkage_matrix = linkage(X_hc_sample, method="ward")

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(
    linkage_matrix, ax=ax,
    truncate_mode="lastp", p=30,
    leaf_rotation=90, leaf_font_size=8,
    show_contracted=True
)
ax.set_title("Hierarchical Clustering Dendrogram (Ward linkage, 200-point sample)", fontsize=11)
ax.set_xlabel("Sample index (or cluster size in brackets)")
ax.set_ylabel("Distance")
plt.tight_layout()
plt.savefig(FIGURES / "20_dendrogram.png", bbox_inches="tight")
plt.show()

print("Interpretation:")
print("The dendrogram shows merge distances on the y-axis. A large jump in merge")
print("distance (a long vertical line before two clusters merge) suggests that")
print("cutting just below that jump gives a natural number of clusters.")

In [ ]:
# apply hierarchical clustering to full data and compare with K-Means
hc = AgglomerativeClustering(n_clusters=best_k_cl, linkage="ward")
df["hc_cluster"] = hc.fit_predict(X_cl_sc)

print(f"Hierarchical Clustering (k={best_k_cl}, Ward) — Cluster Profiles")
print("-" * 65)
hc_profile = df.groupby("hc_cluster")[CLUSTER_FEATURES + ["is_hype_period"]].mean().round(4)
display(hc_profile)

# compare K-Means vs HC cluster assignments
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(df["kmeans_cluster"], df["hc_cluster"])
print(f"\nAdjusted Rand Index (K-Means vs HC): {ari:.4f}")
print("(ARI=1.0 means identical assignments, ARI≈0 means no agreement)")

hc_sil = silhouette_score(X_cl_sc, df["hc_cluster"])
km_sil = silhouette_score(X_cl_sc, df["kmeans_cluster"])
print(f"\nSilhouette — K-Means: {km_sil:.4f}  |  Hierarchical: {hc_sil:.4f}")
print("Higher silhouette = tighter, better-separated clusters.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for cid in sorted(df["kmeans_cluster"].unique()):
    sub = df[df["kmeans_cluster"] == cid]
    axes[0].scatter(sub["hype_index"], sub["NVDA_return"],
                    s=8, alpha=0.45, color=PALETTE[cid % 4], label=f"Cluster {cid}")
axes[0].set_title(f"K-Means (k={best_k_cl})")
axes[0].set_xlabel("Hype Index"); axes[0].set_ylabel("NVDA Return")
axes[0].legend(markerscale=3)

for cid in sorted(df["hc_cluster"].unique()):
    sub = df[df["hc_cluster"] == cid]
    axes[1].scatter(sub["hype_index"], sub["NVDA_return"],
                    s=8, alpha=0.45, color=PALETTE[cid % 4], label=f"Cluster {cid}")
axes[1].set_title(f"Hierarchical Clustering (k={best_k_cl}, Ward)")
axes[1].set_xlabel("Hype Index"); axes[1].set_ylabel("NVDA Return")
axes[1].legend(markerscale=3)

fig.suptitle("K-Means vs Hierarchical Clustering — Hype Index vs NVDA Return",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES / "21_clustering_comparison.png", bbox_inches="tight")
plt.show()

## 9. Model Comparison

All classification models evaluated on the same test set with the same CV protocol.

In [ ]:
summary_cols = ["Model", "CV Acc (mean)", "CV Acc (std)", "CV AUC (mean)", "CV AUC (std)", "Test Acc", "Test AUC"]
comp_df = (
    pd.DataFrame([{k: r[k] for k in summary_cols} for r in all_results])
    .set_index("Model")
    .sort_values("CV AUC (mean)", ascending=False)
)

print("Classification Model Comparison")
print("=" * 70)
display(comp_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CV AUC with error bars
models = comp_df.index.tolist()
x = np.arange(len(models))
axes[0].barh(x, comp_df["CV AUC (mean)"], xerr=comp_df["CV AUC (std)"],
             color="#1f77b4", alpha=0.8, capsize=4)
axes[0].set_yticks(x); axes[0].set_yticklabels(models)
axes[0].axvline(0.5, color="gray", linestyle="--", linewidth=0.8)
axes[0].set_xlabel("CV AUC (mean ± std)")
axes[0].set_title("Cross-Validated AUC")

# test acc vs test AUC
axes[1].scatter(comp_df["Test Acc"], comp_df["Test AUC"], s=80, color="#ff7f0e", zorder=3)
for i, model in enumerate(comp_df.index):
    axes[1].annotate(model, (comp_df.loc[model, "Test Acc"], comp_df.loc[model, "Test AUC"]),
                     fontsize=7.5, xytext=(5, 3), textcoords="offset points")
axes[1].set_xlabel("Test Accuracy"); axes[1].set_ylabel("Test AUC")
axes[1].set_title("Test Accuracy vs Test AUC")

plt.tight_layout()
plt.savefig(FIGURES / "22_model_comparison.png", bbox_inches="tight")
plt.show()

# ROC curves for all models on the same plot
fig, ax = plt.subplots(figsize=(8, 6))
colors_roc = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2"]
for i, r in enumerate(all_results):
    fpr, tpr, _ = roc_curve(y_te_c, r["_y_prob"])
    ax.plot(fpr, tpr, lw=1.8, color=colors_roc[i % len(colors_roc)],
            label=f"{r['Model']} (AUC={r['Test AUC']:.3f})")
ax.plot([0,1],[0,1], "k--", lw=1, label="random")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Models", fontsize=11)
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(FIGURES / "23_roc_all_models.png", bbox_inches="tight")
plt.show()

## 10. Takeaways

**Regression:** Linear models explain very little variance in forward returns (low R²) across all three tickers, and NVDA shows the strongest (though still weak) relationship — consistent with it being the most directly exposed to AI infrastructure demand. Lasso regularization tends to zero out the longer lags, suggesting that very recent hype carries more predictive weight than hype from 20 days ago. The key takeaway here is not that hype is useless — it's that the relationship is too noisy and nonlinear for a linear model to capture reliably. This is consistent with market efficiency at the linear level.

**Classification:** All classification models substantially outperform the random baseline (AUC = 0.5), confirming that stock features do contain a signal about whether a high-hype regime is ongoing. Ensemble methods consistently lead — gradient boosting and random forest outperform the single decision tree and logistic regression, which is expected: hype regimes involve nonlinear interactions between multiple signals that linear models cannot represent. The voting ensemble often matches or beats any single model, confirming that LR, RF, and GBT make partially independent errors.

**Feature importance:** Random Forest rankings consistently place volatility features alongside the hype index at the top. This confirms that hype periods are fundamentally high-volatility regimes — not just high-return ones — which is exactly what a speculative bubble story would predict.

**Clustering:** Both K-Means and Hierarchical clustering naturally separate the data into regimes that correspond roughly to pre-ChatGPT calm, the hype surge (post-November 2022), and the post-peak normalization. The cluster with the highest hype fraction shows elevated NVDA volatility and positive average returns, but the relationship is noisy — consistent with a sentiment-driven rather than fundamentals-driven price movement.

**Overall conclusion:** AI hype, measured through Google Trends and Reddit signals, is correlated with elevated stock activity in NVDA, MSFT, and AAPL. The relationship is nonlinear and regime-dependent — ensemble methods can partially exploit it, but simple linear models largely cannot. The clustering analysis suggests the market has distinct behavioral regimes that align with the hype cycle, reinforcing the core thesis of this project.